In [4]:
!pip install ucimlrepo kagglehub umap-learn scikit-learn --quiet
!pip install tensorflow



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 721.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.8 MB/s eta 0:00:00


In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error
from ucimlrepo import fetch_ucirepo
import tensorflow as tf
import umap
import os


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [6]:
def umap_reconstruction_mse(X, n_components_list, n_neighbors=5, random_state=42):
    """
    Apply UMAP for dimensionality reduction and approximate reconstruction using k-NN.

    Parameters:
        X (numpy.ndarray): Original dataset (samples x features)
        n_components_list (list): List of target dimensions to reduce to
        n_neighbors (int): Number of neighbors for k-NN reconstruction
        random_state (int): Random seed for reproducibility

    Returns:
        pd.DataFrame: Dataset with n_components and reconstruction MSE
    """
    results = []

    # Standardize data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    for n_comp in n_components_list:
        # Apply UMAP
        reducer = umap.UMAP(n_components=n_comp, n_neighbors=15, min_dist=0.1,
                            metric='euclidean', random_state=random_state)
        X_embedded = reducer.fit_transform(X_scaled)

        # Approximate inverse transformation using k-NN
        knn = KNeighborsRegressor(n_neighbors=n_neighbors)
        knn.fit(X_embedded, X_scaled)
        X_reconstructed = knn.predict(X_embedded)

        # Compute MSE
        mse = mean_squared_error(X_scaled, X_reconstructed)

        results.append({
            'UMAP Components': n_comp,
            'Reconstruction MSE': mse
        })

    return pd.DataFrame(results)


In [7]:
# Reduction levels from your table
components_wine = [1, 2, 5, 8, 9]        # Wine: 11 features
components_breast = [3, 7, 15, 22, 27]  # Breast Cancer: 30 features
components_mnist = [78, 196, 392, 588, 705]  # MNIST: 784 features


In [8]:
# Load Wine dataset
wine_data = fetch_ucirepo(id=186)  # Wine Quality dataset
X_wine = wine_data.data.features

# Apply UMAP + k-NN reconstruction
wine_umap_results = umap_reconstruction_mse(X_wine, components_wine)
wine_umap_results['Dataset'] = 'Wine'
wine_umap_results


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,UMAP Components,Reconstruction MSE,Dataset
0,1,0.204914,Wine
1,2,0.097897,Wine
2,5,0.067124,Wine
3,8,0.066267,Wine
4,9,0.065798,Wine


In [9]:
# Load Breast Cancer dataset
breast_data = fetch_ucirepo(id=17)
X_breast = breast_data.data.features

# Apply UMAP + k-NN reconstruction
breast_umap_results = umap_reconstruction_mse(X_breast, components_breast)
breast_umap_results['Dataset'] = 'Breast Cancer'
breast_umap_results


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,UMAP Components,Reconstruction MSE,Dataset
0,3,0.187396,Breast Cancer
1,7,0.181644,Breast Cancer
2,15,0.183151,Breast Cancer
3,22,0.174854,Breast Cancer
4,27,0.180918,Breast Cancer


In [10]:
# Load MNIST dataset
(X_train, _), (_, _) = tf.keras.datasets.mnist.load_data()
X_mnist = X_train.reshape(X_train.shape[0], -1)  # Flatten to (60000, 784)

# Apply UMAP + k-NN reconstruction
mnist_umap_results = umap_reconstruction_mse(X_mnist, components_mnist)
mnist_umap_results['Dataset'] = 'MNIST'
mnist_umap_results


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,UMAP Components,Reconstruction MSE,Dataset
0,78,0.268582,MNIST
1,196,0.268033,MNIST
2,392,0.269847,MNIST
3,588,0.269337,MNIST
4,705,0.268611,MNIST


In [11]:
# Combine all results into one dataframe
all_umap_results = pd.concat([wine_umap_results, breast_umap_results, mnist_umap_results], ignore_index=True)
all_umap_results


,UMAP Components,Reconstruction MSE,Dataset
0,1,0.204914,Wine
1,2,0.097897,Wine
2,5,0.067124,Wine
3,8,0.066267,Wine
4,9,0.065798,Wine
5,3,0.187396,Breast Cancer
6,7,0.181644,Breast Cancer
7,15,0.183151,Breast Cancer
8,22,0.174854,Breast Cancer
9,27,0.180918,Breast Cancer
